# Hotel Booking Cancellation & Overbooking Decision
## 07 — Detailed Model Evaluation and Threshold Analysis

Baseline results identified **Random Forest** as the strongest overall candidate (Accuracy 0.8958, Precision 0.8960, Recall 0.8132, F1 0.8526, ROC-AUC 0.9594, PR-AUC 0.9420). HistGradientBoosting was second. This notebook compares them and studies Random Forest thresholds.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,
roc_auc_score,average_precision_score,roc_curve,precision_recall_curve,
confusion_matrix,ConfusionMatrixDisplay)
RANDOM_STATE=42

## 2. Load Data and Recreate Features

In [ ]:
df=pd.read_csv('../data/processed/hotel_bookings_cleaned.csv')
df_model=df.copy()
df_model['total_nights']=df_model['stays_in_weekend_nights']+df_model['stays_in_week_nights']
df_model['total_guests']=df_model['adults']+df_model['children']+df_model['babies']
df_model['total_previous_bookings']=df_model['previous_cancellations']+df_model['previous_bookings_not_canceled']
df_model['has_special_request']=(df_model['total_of_special_requests']>0).astype(int)
df_model['room_type_changed']=(df_model['reserved_room_type']!=df_model['assigned_room_type']).astype(int)
df_model=df_model.drop(columns=['stays_in_weekend_nights','stays_in_week_nights','adults','children','babies'])
print(df_model.shape)

## 3. Same Stratified Train/Test Split

In [ ]:
X=df_model.drop(columns=['is_canceled']); y=df_model['is_canceled']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE,stratify=y)
num=X_train.select_dtypes(include=np.number).columns.tolist()
cat=X_train.select_dtypes(include=['object','category']).columns.tolist()
print(X_train.shape,X_test.shape)

## 4. Preprocessing

In [ ]:
preprocessor=ColumnTransformer([
('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),num),
('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
boost_preprocessor=ColumnTransformer([
('num',SimpleImputer(strategy='median'),num),
('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))]),cat)])

## 5. Train Top Two Candidates

In [ ]:
candidates={
'Random Forest':Pipeline([('preprocessor',preprocessor),('model',RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1))]),
'HistGradientBoosting':Pipeline([('preprocessor',boost_preprocessor),('model',HistGradientBoostingClassifier(random_state=RANDOM_STATE))])}
probs={}; preds={}
for name,model in candidates.items():
    print('Training',name)
    model.fit(X_train,y_train)
    probs[name]=model.predict_proba(X_test)[:,1]
    preds[name]=model.predict(X_test)
print('Done')

## 6. Candidate Metrics

In [ ]:
rows=[]
for name in candidates:
    p=preds[name]; pr=probs[name]
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,p),'Precision':precision_score(y_test,p),
    'Recall':recall_score(y_test,p),'F1':f1_score(y_test,p),'ROC_AUC':roc_auc_score(y_test,pr),
    'PR_AUC':average_precision_score(y_test,pr)})
candidate_results=pd.DataFrame(rows)
candidate_results.round(4)

## 7. ROC Curves

In [ ]:
plt.figure(figsize=(8,6))
for name,pr in probs.items():
    fpr,tpr,_=roc_curve(y_test,pr)
    plt.plot(fpr,tpr,label=f"{name} (AUC={roc_auc_score(y_test,pr):.4f})")
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Comparison'); plt.legend(); plt.show()

## 8. Precision-Recall Curves

In [ ]:
plt.figure(figsize=(8,6))
for name,pr in probs.items():
    precision,recall,_=precision_recall_curve(y_test,pr)
    plt.plot(recall,precision,label=f"{name} (AP={average_precision_score(y_test,pr):.4f})")
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Comparison'); plt.legend(); plt.show()

## 9. Random Forest Cancellation Probabilities

In [ ]:
rf_prob=probs['Random Forest']
plt.figure(figsize=(9,5))
plt.hist(rf_prob[y_test.to_numpy()==0],bins=30,alpha=.6,label='Not Cancelled')
plt.hist(rf_prob[y_test.to_numpy()==1],bins=30,alpha=.6,label='Cancelled')
plt.xlabel('Predicted Cancellation Probability'); plt.ylabel('Bookings')
plt.title('Random Forest Probability Distribution'); plt.legend(); plt.show()

## 10. Threshold Analysis

At threshold 0.50, probability >= 0.50 predicts cancellation. Lowering the threshold usually increases recall but may increase false positives; raising it usually increases precision but can miss more cancellations.

In [ ]:
threshold_rows=[]
for t in np.arange(.20,.81,.05):
    p=(rf_prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_test,p).ravel()
    threshold_rows.append({'Threshold':round(t,2),'Accuracy':accuracy_score(y_test,p),
    'Precision':precision_score(y_test,p,zero_division=0),'Recall':recall_score(y_test,p,zero_division=0),
    'F1':f1_score(y_test,p,zero_division=0),'False_Positives':fp,'False_Negatives':fn})
threshold_df=pd.DataFrame(threshold_rows)
threshold_df.round(4)

## 11. Threshold Trade-off Plot

In [ ]:
plt.figure(figsize=(9,6))
for metric in ['Precision','Recall','F1']:
    plt.plot(threshold_df['Threshold'],threshold_df[metric],marker='o',label=metric)
plt.xlabel('Threshold'); plt.ylabel('Score'); plt.title('Random Forest Threshold Trade-offs'); plt.legend(); plt.show()

## 12. Default Threshold Confusion Matrix

In [ ]:
p=(rf_prob>=.50).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test,p),display_labels=['Not Cancelled','Cancelled']).plot(values_format='d')
plt.title('Random Forest - Threshold 0.50'); plt.show()

## 13. Expected Cancellations from Probability Sum

In [ ]:
expected=rf_prob.sum()
actual=y_test.sum()
print('Test bookings:',len(y_test))
print('Expected cancellations:',round(expected,2))
print('Actual cancellations:',int(actual))
print('Difference:',round(expected-actual,2))

### Why sum probabilities?
If five bookings have probabilities 0.80, 0.70, 0.60, 0.40 and 0.20, the expected cancellations are 2.70. This is an expected count across the group, not a claim that exactly 2.7 people can cancel.

## 14. Current Selection

In [ ]:
print(candidate_results.round(4))
print('\nCurrent strongest candidate: Random Forest')

## Conclusion

Random Forest remains the strongest overall baseline candidate based on its combination of Accuracy, Precision, F1, ROC-AUC and PR-AUC. Threshold analysis shows why the default 0.50 cutoff is not automatically the best business decision.

### Next Step
Introduce the cost of an empty room versus the cost of excessive overbooking, then combine those business costs with cancellation probabilities to evaluate safer overbooking decisions.